# Retinal Disease Detection Using Deep Learning on OCT Images
## Part 1: Dataset Exploration, Preprocessing & Neural Network Fundamentals
**Domain:** Healthcare & Life Sciences – Medical Imaging (Ophthalmology)
**Classes:** CNV, DME, DRUSEN, NORMAL


In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch

from src.config import config
from src.dataset import build_dataloaders, get_transforms
from src.nn_fundamentals import Activations, LossFunctions, ScratchNeuralNetwork
from src.utils.seed import seed_everything

seed_everything(config.seed)
print(f'PyTorch Version: {torch.__version__} | Hardware Acceleration: {config.device}')


---
### 1. Dataset Collection and Class Distribution Analysis
Analyzing sample counts and class balance across the 4 diagnostic categories.


In [ ]:
sample_counts = {}
for cls in config.classes:
    cls_dir = config.samples_dir / cls
    images = list(cls_dir.glob('*.jpeg')) + list(cls_dir.glob('*.png'))
    sample_counts[cls] = len(images)

plt.figure(figsize=(8, 4))
sns.barplot(x=list(sample_counts.keys()), y=list(sample_counts.values()), palette='crest')
plt.title('OCT Dataset Class Balance Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Diagnostic Category')
plt.ylabel('Number of Images')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


---
### 2. Sample Image Visualization across Pathologies
Visualizing the cross-sectional anatomy for Normal, CNV, DME, and Drusen.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for idx, cls in enumerate(config.classes):
    cls_dir = config.samples_dir / cls
    sample_img_path = list(cls_dir.glob('*.*'))[0]
    img = Image.open(sample_img_path).convert('RGB')
    axes[idx].imshow(img)
    axes[idx].set_title(f'{cls}\n{sample_img_path.name}', fontsize=11, fontweight='bold')
    axes[idx].axis('off')
plt.tight_layout()
plt.show()


---
### 3. Neural Network Fundamentals from Scratch (NumPy)
Testing activation functions (ReLU, Sigmoid, Tanh, Softmax) and analytical gradient backpropagation.


In [ ]:
# Generate toy synthetic diagnostic features
np.random.seed(42)
N_samples = 200
X_toy = np.random.randn(N_samples, 8)
y_labels = np.random.randint(0, 4, size=N_samples)
y_one_hot = np.zeros((N_samples, 4))
y_one_hot[np.arange(N_samples), y_labels] = 1.0

mlp = ScratchNeuralNetwork(input_dim=8, hidden_dim=16, output_dim=4, activation='relu', learning_rate=0.05)
history = mlp.fit(X_toy, y_one_hot, epochs=60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['loss'], 'r-', label='Cross-Entropy Loss', linewidth=2)
ax1.set_title('NumPy MLP Training Loss Convergence')
ax1.set_xlabel('Epoch')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.plot(history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
ax2.set_title('NumPy MLP Accuracy')
ax2.set_xlabel('Epoch')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()
plt.show()
